In [37]:
!pip install transformers optuna accelerate python-whois

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import DistilBertModel, DistilBertTokenizer
import pandas as pd
import numpy as np
import math, re, datetime, whois, time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from urllib.parse import urlparse
from collections import Counter

# --- 1. IDENTITY LOGIC (WHOIS + HUMAN READABLE DATE) ---
def get_domain_identity(url):
    try:
        hostname = urlparse(url).hostname
        if not hostname: return -1, "Unknown"

        domain = ".".join(hostname.split(".")[-2:])
        w = whois.whois(domain)

        creation_date = w.creation_date
        if isinstance(creation_date, list): creation_date = creation_date[0]

        if creation_date:
            days_old = (datetime.datetime.now() - creation_date).days
            date_str = creation_date.strftime('%Y-%m-%d (Year: %Y)')
            return days_old, date_str

        return -1, "Domain Record Not Found"
    except:
        return -1, "Private/Hidden Registration"

# --- 2. EXPERT LOGIC (Statistical + Identity Age) ---
def extract_master_features_v5(url, age_val=None):
    url_str = str(url).lower()
    parsed = urlparse(url_str)
    domain = parsed.netloc

    has_https = 1 if url_str.startswith('https') else 0
    at_count = url_str.count('@')
    pct_count = url_str.count('%')

    # Interaction: HTTPS + Danger symbols (The Bias Breaker)
    danger_score = (at_count * 10.0) + (pct_count * 10.0)
    if has_https and danger_score > 0: danger_score += 15.0

    f = [
        len(url_str), len(domain), url_str.count('.'),
        at_count, pct_count, danger_score,
        sum(c.isdigit() for c in url_str), has_https,
        age_val if age_val is not None else 0 # Numerical Age for the AI
    ]
    return f

print("Step 1 Complete: Identity Engine and Date Formatting Ready.")

Step 1 Complete: Identity Engine and Date Formatting Ready.


In [38]:
# --- 3. DATA PREPARATION ---
df1 = pd.read_csv('phishing_features.csv')
df2 = pd.read_csv('processed_raw_data.csv')
master_df = pd.concat([df1, df2], ignore_index=True).drop_duplicates(subset=['url'])

# Balanced Subset
phish_df = master_df[master_df['label'] == 1].sample(15000, random_state=42)
legit_df = master_df[master_df['label'] == 0].sample(15000, random_state=42)
balanced_df = pd.concat([phish_df, legit_df]).sample(frac=1).reset_index(drop=True)

# Generate features with synthetic WHOIS ages (in days)
expert_raw = []
for i, row in balanced_df.iterrows():
    # Phish: 0-10 days old. Safe: 1000-8000 days old.
    age = np.random.randint(0, 10) if row['label'] == 1 else np.random.randint(1000, 8000)
    expert_raw.append(extract_master_features_v5(row['url'], age_val=age))

scaler = MinMaxScaler()
expert_scaled = scaler.fit_transform(np.array(expert_raw))

# Tokenizer/Dataset Setup (Same as V4)
tokenizer_bert = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
chars = "abcdefghijklmnopqrstuvwxyz0123456789-._~:/?#[]@!$&'()*+,;=%"
char_to_int = {char: i + 1 for i, char in enumerate(chars)}

def encode_chars(url):
    res = [char_to_int.get(c, 0) for c in str(url)[:128]]
    return res + [0] * (128 - len(res))

class IdentityDataset(Dataset):
    def __init__(self, urls, feats, labels):
        self.urls = urls.tolist(); self.feats = torch.tensor(feats, dtype=torch.float)
        self.labels = torch.tensor(labels.tolist(), dtype=torch.float)
        self.enc = tokenizer_bert(self.urls, truncation=True, padding='max_length', max_length=128, return_tensors="pt")
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {'ids': self.enc['input_ids'][idx], 'mask': self.enc['attention_mask'][idx],
                'char': torch.tensor(encode_chars(self.urls[idx]), dtype=torch.long),
                'feat': self.feats[idx], 'label': self.labels[idx]}

train_loader = DataLoader(IdentityDataset(balanced_df['url'], expert_scaled, balanced_df['label']), batch_size=16, shuffle=True)
print("Step 2 Complete: Adversarial Shuffled Data Ready.")

Step 2 Complete: Adversarial Shuffled Data Ready.


In [39]:
class FinalIdentityModel(nn.Module):
    def __init__(self):
        super(FinalIdentityModel, self).__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.char_embed = nn.Embedding(75, 64)
        self.conv = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.lstm = nn.LSTM(128, 64, bidirectional=True, batch_first=True)
        self.expert_fc = nn.Sequential(nn.Linear(9, 256), nn.ReLU(), nn.Dropout(0.2)) # 9 Expert Features

        self.fusion = nn.Sequential(
            nn.Linear(768 + 128 + 256, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1),
            nn.Sigmoid()
        )

    def forward(self, ids, mask, char_seq, expert_feat):
        b_out = F.dropout(self.bert(ids, attention_mask=mask)[0][:, 0, :], p=0.4)
        x_char = F.relu(self.conv(self.char_embed(char_seq).transpose(1, 2))).transpose(1, 2)
        _, (h_n, _) = self.lstm(x_char)
        char_out = torch.cat((h_n[0], h_n[1]), dim=1)
        exp_out = self.expert_fc(expert_feat)
        return self.fusion(torch.cat((b_out, char_out, exp_out), dim=1))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FinalIdentityModel().to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCELoss()

print("Step 3: Model Built. Starting Training...")
model.train()
for epoch in range(3):
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch['ids'].to(device), batch['mask'].to(device), batch['char'].to(device), batch['feat'].to(device))
        loss = criterion(out.squeeze(), batch['label'].to(device))
        loss.backward(); optimizer.step()
    print(f"Epoch {epoch+1} Done.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step 3: Model Built. Starting Training...
Epoch 1 Done.
Epoch 2 Done.
Epoch 3 Done.


In [57]:
def scan_with_real_date(url):
    model.eval()
    print(f"\n[AI SCANNER] Analysis for: {url}")

    # 1. LIVE IDENTITY RECORD CHECK
    age_days, real_date = get_domain_identity(url)
    print(f"[*] IDENTITY VERIFIED: Created on {real_date}")
    if age_days > 0:
        print(f"[*] DOMAIN MATURITY: {age_days} days old")

    # 2. EXTRACT AI FEATURES
    feats = extract_master_features_v5(url, age_val=age_days)
    danger_symbols_score = feats[5]

    with torch.no_grad():
        e_feat = torch.tensor(scaler.transform([feats]), dtype=torch.float).to(device)
        char_s = torch.tensor([[char_to_int.get(c, 0) for c in url[:128]] + [0]*max(0, 128-len(url))], dtype=torch.long).to(device)
        b_in = tokenizer_bert(url, return_tensors="pt", truncation=True, padding='max_length', max_length=128).to(device)

        prob = model(b_in['input_ids'], b_in['attention_mask'], char_s, e_feat).item()

        # FINAL PROTECTION LOGIC
        # If domain is brand new (less than a month) but trying to mimic a big brand
        if 0 < age_days < 30:
            prob = max(prob, 0.96)
            print("[!] WARNING: This domain was registered very recently.")

        if danger_symbols_score > 10.0:
            prob = max(prob, 0.99)
            print("[!] WARNING: Suspect character obfuscation (@/%) detected.")

        res = "!!! PHISHING !!!" if prob > 0.2 else "SAFE (LEGITIMATE)"
        print(f"FINAL RESULT: {res} ({prob*100:.2f}% Probability)")
        print("-" * 65)

# --- THE GRAND UNBEATABLE TEST ---
scan_with_real_date("https://www.hotstar.com/in/home")
scan_with_real_date("https://www.netflix.com/in/")
scan_with_real_date("http://www.netflix.com/in/")
scan_with_real_date("https://www.go%oooo@oooogIe.com")


[AI SCANNER] Analysis for: https://www.hotstar.com/in/home


ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out


[*] IDENTITY VERIFIED: Created on Private/Hidden Registration
FINAL RESULT: SAFE (LEGITIMATE) (0.11% Probability)
-----------------------------------------------------------------

[AI SCANNER] Analysis for: https://www.netflix.com/in/
[*] IDENTITY VERIFIED: Created on Private/Hidden Registration
FINAL RESULT: SAFE (LEGITIMATE) (1.99% Probability)
-----------------------------------------------------------------

[AI SCANNER] Analysis for: http://www.netflix.com/in/
[*] IDENTITY VERIFIED: Created on Private/Hidden Registration
FINAL RESULT: !!! PHISHING !!! (100.00% Probability)
-----------------------------------------------------------------

[AI SCANNER] Analysis for: https://www.go%oooo@oooogIe.com
[*] IDENTITY VERIFIED: Created on Private/Hidden Registration
[!] WARNING: Suspect character obfuscation (@/%) detected.
FINAL RESULT: !!! PHISHING !!! (99.00% Probability)
-----------------------------------------------------------------


In [58]:

scan_with_real_date("https://urlhaus.abuse.ch/host/pub-3bc1de741f8149f49bdbafa703067f24.r2.dev/")
scan_with_real_date("https://pub-3bc1de741f8149f49bdbafa703067f24.r2.dev/ha3.png")
scan_with_real_date("https://www.easysyncr.io/6KD2ly91")
scan_with_real_date("https://www.5movierulz.claims/movies/page/1")
scan_with_real_date("https://kelfluxor.oakumsenile.in.net/verification.google")
scan_with_real_date("https://urlhaus.abuse.ch/url/3792042/")
scan_with_real_date("https://ultimategetawaysapps.org.in/stor...")
scan_with_real_date("http://bluetelecombolivia.com/")
scan_with_real_date("https://login.galaxynotenews.com/?login=...")


[AI SCANNER] Analysis for: https://urlhaus.abuse.ch/host/pub-3bc1de741f8149f49bdbafa703067f24.r2.dev/


ERROR:whois.whois:Error trying to connect to socket: closing socket - [Errno -2] Name or service not known


[*] IDENTITY VERIFIED: Created on Domain Record Not Found
FINAL RESULT: !!! PHISHING !!! (100.00% Probability)
-----------------------------------------------------------------

[AI SCANNER] Analysis for: https://pub-3bc1de741f8149f49bdbafa703067f24.r2.dev/ha3.png
[*] IDENTITY VERIFIED: Created on Domain Record Not Found
FINAL RESULT: !!! PHISHING !!! (100.00% Probability)
-----------------------------------------------------------------

[AI SCANNER] Analysis for: https://www.easysyncr.io/6KD2ly91


ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out


[*] IDENTITY VERIFIED: Created on Domain Record Not Found
FINAL RESULT: !!! PHISHING !!! (97.13% Probability)
-----------------------------------------------------------------

[AI SCANNER] Analysis for: https://www.5movierulz.claims/movies/page/1
[*] IDENTITY VERIFIED: Created on Private/Hidden Registration
FINAL RESULT: SAFE (LEGITIMATE) (0.02% Probability)
-----------------------------------------------------------------

[AI SCANNER] Analysis for: https://kelfluxor.oakumsenile.in.net/verification.google
[*] IDENTITY VERIFIED: Created on Private/Hidden Registration
FINAL RESULT: !!! PHISHING !!! (100.00% Probability)
-----------------------------------------------------------------

[AI SCANNER] Analysis for: https://urlhaus.abuse.ch/url/3792042/
[*] IDENTITY VERIFIED: Created on Domain Record Not Found
FINAL RESULT: !!! PHISHING !!! (100.00% Probability)
-----------------------------------------------------------------

[AI SCANNER] Analysis for: https://ultimategetawaysapps.org.in

ERROR:whois.whois:Error trying to connect to socket: closing socket - [Errno -2] Name or service not known


[*] IDENTITY VERIFIED: Created on Private/Hidden Registration
FINAL RESULT: !!! PHISHING !!! (99.93% Probability)
-----------------------------------------------------------------

[AI SCANNER] Analysis for: http://bluetelecombolivia.com/
[*] IDENTITY VERIFIED: Created on Private/Hidden Registration
FINAL RESULT: !!! PHISHING !!! (100.00% Probability)
-----------------------------------------------------------------

[AI SCANNER] Analysis for: https://login.galaxynotenews.com/?login=...
[*] IDENTITY VERIFIED: Created on Private/Hidden Registration
FINAL RESULT: !!! PHISHING !!! (100.00% Probability)
-----------------------------------------------------------------


In [44]:
import joblib, json, torch

# 1. Save Model Weights
torch.save(model.state_dict(), 'asra_model.pth')

# 2. Save the Scaler
joblib.dump(scaler, 'asra_scaler.pkl')

# 3. Save Character Mapping for the CNN branch
with open('asra_chars.json', 'w') as f:
    json.dump(char_to_int, f)

print("ASRA Brain Exported! Download: asra_model.pth, asra_scaler.pkl, asra_chars.json")

ASRA Brain Exported! Download: asra_model.pth, asra_scaler.pkl, asra_chars.json
